In [ ]:
#*************************************************************************
#   > Filename    : lr_gcn_cora_bit.py
#   > Description : Quatization process when the number of qGconv layer is equal to two.
#*************************************************************************


In [1]:

#*************************************************************************
from ast import parse
import os
import torch
import torch.nn as nn
from torch.nn import Linear, Sequential, ReLU, Identity, BatchNorm1d as BN
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree,remove_self_loops
from torch_geometric.data import Data
from torch_geometric.datasets import TUDataset,Planetoid,GNNBenchmarkDataset
from torch_geometric.loader import DataLoader
from torch_scatter import scatter_mean
from torch.autograd.function import InplaceFunction
from torch_geometric.nn import GCNConv,GINConv,global_mean_pool,global_add_pool, TopKPooling
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import random
import numpy as np
from quantize_function.u_quant_gc_bit_debug import *
import argparse


import statistics as stat
from tabulate import tabulate


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
import itertools
import os

os.environ["DGLBACKEND"] = "pytorch"

import copy
import dgl
import dgl.data
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from sklearn.metrics import roc_auc_score
from torchprofile import profile_macs


def get_model_macs(model, inputs) -> int:
    '''
    MACs are a common metric to measure the computational complexity 
    of deep neural networks, as they reflect the number of arithmetic
    operations involved in the forward pass of the model
    '''
    return profile_macs(model, inputs)


def get_sparsity(tensor: torch.Tensor) -> float:
    """
    calculate the sparsity of the given tensor
        sparsity = #zeros / #elements = 1 - #nonzeros / #elements
    """
    return 1 - float(tensor.count_nonzero()) / tensor.numel()


def get_model_sparsity(model: nn.Module) -> float:
    """
    calculate the sparsity of the given model
        sparsity = #zeros / #elements = 1 - #nonzeros / #elements
    """
    num_nonzeros, num_elements = 0, 0
    for param in model.parameters():
        num_nonzeros += param.count_nonzero()
        num_elements += param.numel()
    return 1 - float(num_nonzeros) / num_elements

def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements


def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=False) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB



# Function For Quatization

In [3]:
def paras_group(model):
    all_params = model.parameters()
    weight_paras=[]
    quant_paras_bit_weight = []
    quant_paras_bit_fea = []
    quant_paras_scale_weight = []
    quant_paras_scale_fea = []
    quant_paras_scale_xw = []
    quant_paras_bit_xw = []
    other_paras = []
    for name,para in model.named_parameters():
        if('quant' in name and 'bit' in name and 'weight' in name):
            quant_paras_bit_weight+=[para]
            # para.requires_grad = False
        elif('quant' in name and 'bit' in name and 'fea' in name):
            quant_paras_bit_fea+=[para]
        elif('quant' in name and 'bit' not in name and 'weight' in name):
            quant_paras_scale_weight+=[para]
            # para.requires_grad = False
        elif('quant' in name and 'bit' not in name and 'fea' in name):
            quant_paras_scale_fea+=[para]
        elif('xw'in name and 'q' in name and 'bit' not in name):
            quant_paras_scale_xw+=[para]
        elif('xw'in name and 'q' in name and 'bit' in name):
            quant_paras_bit_xw+=[para]
        elif('weight' in name and 'quant' not in name ):
            weight_paras+=[para]
    params_id = list(map(id,quant_paras_bit_fea))+list(map(id,quant_paras_bit_weight))+list(map(id,quant_paras_scale_weight))+list(map(id,quant_paras_scale_fea))+list(map(id,weight_paras))\
    +list(map(id,quant_paras_scale_xw))+list(map(id,quant_paras_bit_xw))
    other_paras = list(filter(lambda p: id(p) not in params_id, all_params))
    return weight_paras,quant_paras_bit_weight,quant_paras_bit_fea,quant_paras_scale_weight,quant_paras_scale_fea,quant_paras_scale_xw,quant_paras_bit_xw,other_paras




In [4]:
def setup_seed(seed):
     torch.manual_seed(seed)
     torch.cuda.manual_seed_all(seed)
     np.random.seed(seed)
     random.seed(seed)
    #  torch.backends.cudnn.deterministic = True

def parameter_stastic(model,dataset,hidden_units):
    w_Byte = 0
    a_Byte = 0
    for name, par in model.named_parameters():
        if(('bit' in name)&('weight' in name)):
            if('conv1' in name):
                scale = dataset.num_node_features
            else:
                scale = hidden_units
            par = torch.floor(par)
            w_Byte = scale*par.sum()/8./1024.+w_Byte
        elif(('bit' in name)&('fea' in name)):
            if('conv1' in name):
                a_scale = 0
            else:
                a_scale = hidden_units
            # a_scale = dataset.data.num_nodes
            par = torch.floor(par)
            a_Byte = a_scale*par.sum()/8./1024.+a_Byte
    return w_Byte, a_Byte



In [5]:
def load_checkpoint(model, checkpoint):
    if checkpoint != 'No':
        print("loading checkpoint...")
        model_dict = model.state_dict()
        modelCheckpoint = torch.load(checkpoint)
        pretrained_dict = modelCheckpoint['state_dict']
        new_dict = {k: v for k, v in pretrained_dict.items() if (k in model_dict.keys())}
        model_dict.update(new_dict)
        print('Total : {}, update: {}'.format(len(pretrained_dict), len(new_dict)))
        model.load_state_dict(model_dict)
        print("loaded finished!")
    return model

In [6]:
def num_graphs(data):
    if data.batch is not None:
        return data.num_graphs
    else:
        return data.x.size(0)

In [8]:
import sys
import argparse

# Clearing the arguments
sys.argv = ['']


parser = argparse.ArgumentParser()
parser.add_argument('--model',type=str,default='GCN')
parser.add_argument('--gpu_id',type=int,default=0)
parser.add_argument('--act_type',type=str,default='QReLU')
parser.add_argument('--dataset_name',type=str,default='Cora')
parser.add_argument('--num_deg',type=int,default=1000)
parser.add_argument('--num_layers', type=int, default=2)
parser.add_argument('--hidden_units',type=int,default=146)
parser.add_argument('--batch-size',type=int,default=128)
parser.add_argument('--bit',type=int,default=4)
parser.add_argument('--max_epoch',type=int,default=200)
parser.add_argument('--max_cycle',type=int,default=10)
parser.add_argument('--folds',type=int,default=5)
parser.add_argument('--a_loss',type=float,default=0.0004)
parser.add_argument('--init',type=str,default='norm')
parser.add_argument('--weight_decay',type=float,default=0)
parser.add_argument('--lr',type=float,default=0.001)
parser.add_argument('--lr_quant_scale_fea',type=float,default=1e-2)
parser.add_argument('--lr_quant_scale_xw',type=float,default=2e-2)
parser.add_argument('--lr_quant_scale_weight',type=float,default=1e-2)
parser.add_argument('--lr_quant_bit_fea',type=float,default=0.0003)
parser.add_argument('--lr_quant_bit_weight',type=float,default=0.0001) 
parser.add_argument('--lr_step_size',type=int, default=50)
parser.add_argument('--lr_decay_factor',type=float,default=0.5)
parser.add_argument('--lr_schedule_patience',type=int,default=10)

###############################################################
parser.add_argument('--resume',type=bool,default=False)
parser.add_argument('--store_ckpt',type=bool,default=True)
parser.add_argument('--q_qmax',type=str,default='q_4layers_lr_bit_1')
parser.add_argument('--uniform',type=bool,default=True)
parser.add_argument('--use_norm_quant',type=bool,default=True)
parser.add_argument('--quant_method',type=int,default=2)
parser.add_argument('--quant_weight',type=int,default=1)
###############################################################
# The target memory size of nodes features
parser.add_argument('--a_storage',type=float,default=1)
# Path to results
parser.add_argument('--result_folder',type=str,default='result')
# Path to checkpoint
parser.add_argument('--check_folder',type=str,default='checkpoint')
# Path to dataset
parser.add_argument('--path2dataset',type=str,default='/')




args = parser.parse_args()
print(args)

Namespace(model='GCN', gpu_id=0, act_type='QReLU', dataset_name='Cora', num_deg=1000, num_layers=2, hidden_units=146, batch_size=128, bit=4, max_epoch=200, max_cycle=10, folds=5, a_loss=0.0004, init='norm', weight_decay=0, lr=0.001, lr_quant_scale_fea=0.01, lr_quant_scale_xw=0.02, lr_quant_scale_weight=0.01, lr_quant_bit_fea=0.0003, lr_quant_bit_weight=0.0001, lr_step_size=50, lr_decay_factor=0.5, lr_schedule_patience=10, resume=False, store_ckpt=True, q_qmax='q_4layers_lr_bit_1', uniform=True, use_norm_quant=True, quant_method=2, quant_weight=1, a_storage=1, result_folder='result', check_folder='checkpoint', path2dataset='/')


In [9]:
###############################################################
model = args.model
act_type = args.act_type
dataset_name = args.dataset_name
num_layers = args.num_layers
hidden_units=args.hidden_units
bit=args.bit
max_epoch = args.max_epoch
q_qmax = args.q_qmax
quant_method = args.quant_method
resume = args.resume
path2result = args.result_folder+'/'+args.model+'_'+dataset_name
path2check = args.check_folder+'/'+args.model+'_'+dataset_name
if not os.path.exists(path2result):  
    os.makedirs(path2result)
if not os.path.exists(path2check):  
    os.makedirs(path2check)
##############################################################
setup_seed(41)
if(args.resume==True):
    file_name = path2result+'/'+args.model+'_'+str(hidden_units)+'_'+'_on_'+dataset_name+'_'+str(bit)+'bit-'+str(max_epoch)+'.txt'
    if not os.path.exists(file_name):
            with open(file_name, 'w') as f:
                for key, value in vars(args).items():
                    f.write('%s:%s\n'%(key, value))

In [10]:
class ResettableSequential(nn.Sequential):
    def reset_parameters(self):
        for child in self.children():
            if hasattr(child, "reset_parameters"):
                child.reset_parameters()
    def forward(self,input,edge_index):
        for model in self:
            input = model(input,edge_index)[0]
        return input 

class MyGCNConv(MessagePassing):
    def __init__(self, in_channels, out_channels, bit, all_positive=False, add_self_loops=False, num_deg=200,
                para_dict={'alpha_init':0.01,'alpha_std':0.01,'gama_init':0.01,'gama_std':0.01},
                is_q=False, 
                uniform=True,
                init='norm'):
        super().__init__(aggr='add')  # "Add" aggregation.
        self.self_loop = add_self_loops
        self.is_q = is_q
        if(is_q):
            self.lin = QLinear(in_channels, out_channels, num_deg,bit, para_dict=para_dict,all_positive=all_positive,
                                    uniform=uniform,
                                    init=init)
        else:
            self.lin = nn.Linear(in_channels, out_channels, bias=False)
        self.q_xw = u_quant_xw(in_channels,out_channels,bit,alpha_init=0.05,alpha_std=0.02)
        self.bias = torch.nn.Parameter(torch.Tensor(out_channels))
        torch.nn.init.zeros_(self.bias)
        torch.nn.init.xavier_uniform_(self.lin.weight)

    def forward(self, x, edge_index, bit_sum):

        # Add self-loops to the adjacency matrix.
        if(self.self_loop):
            edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # Linearly transform node feature matrix.
        if(self.is_q):
            x,_,bit_sum = self.lin(x, edge_index, bit_sum)
        else:
            x = self.lin(x)
            bit_sum = 0
        # Quantize the result of the XW
        x = x.T
        x = self.q_xw(x,)
        x = x.T

        # Do not need to quantize the normalize matrix
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        out=self.propagate(edge_index, x=x, norm=norm)

        out = out + self.bias
        return out, bit_sum
    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j

class GCNlayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, activation, dropout, batch_norm, residual=True,
                bit=4,
                all_positive=False,
                para_dict={'alpha_init':0.01,'alpha_std':0.01,'gama_init':0.01,'gama_std':0.01},
                num_deg=200,
                add_self_loops=False,
                is_q=False,
                uniform=True,
                init='norm'):
        super().__init__()
        self.residual = residual
        self.batch_norm = batch_norm
        if(in_dim!=out_dim):
            self.residual = False
        self.bn = nn.BatchNorm1d(out_dim)
        self.activation = activation
        self.dropout = nn.Dropout(dropout)
        self.conv = MyGCNConv(in_dim, out_dim,bit=bit,
                                all_positive=all_positive,add_self_loops=add_self_loops, 
                                is_q=is_q,
                                num_deg=num_deg,
                                para_dict=para_dict,
                                uniform=uniform,
                                init=init)

    def forward(self,x, edge_index,bit_sum):
        x_in = x.clone()
        x,bit_sum = self.conv(x,edge_index,bit_sum)
        if self.batch_norm:
            x = self.bn(x)
        if self.activation:
            x = self.activation(x)
        if self.residual:
            x = x + x_in
        
        x = self.dropout(x)
        return x, bit_sum

class GCN(torch.nn.Module):
    def __init__(self,dataset, hidden_units,num_layers,dropout=0,batch_norm=True,residual=True,bit=4,add_self_loops=False,
                is_q=False,
                num_deg = 200,
                uniform = False,
                init='norm'
                ):
        super().__init__()
        self.is_q = is_q
        para_list=[[{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'gama_init':0.70,'gama_std':0.1}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'gama_init':0.6,'gama_std':0.7}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'gama_init':0.76,'gama_std':0.68}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'gama_init':0.6,'gama_std':0.5}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1}]]
        if(is_q==True):
            self.embeding = QLinear(dataset.num_features,hidden_units, num_deg, bit,para_dict=para_list[0][0], all_positive=False,
                                    quant_fea=True,
                                    uniform=uniform,
                                    init=init)
        else:
            self.embeding = nn.Linear(dataset.num_features,hidden_units,bias=True)
        self.convs = torch.nn.ModuleList()
        for i in range(num_layers):
            self.convs.append(GCNlayer(hidden_units,hidden_units,F.relu,dropout=0,
                                        batch_norm=batch_norm,
                                        residual=residual,
                                        bit=bit,
                                        all_positive=False,
                                        add_self_loops=add_self_loops,
                                        is_q=is_q,
                                        uniform=uniform,
                                        para_dict=para_list[i+1][0],
                                        num_deg=num_deg,
                                        init=init
                                        ))
        # The readout MLP
        if(is_q):
            self.lin1 = QLinear(hidden_units,hidden_units, num_deg, bit,para_dict=para_list[-2][0], all_positive=False,
                                    quant_fea=True,
                                    uniform=uniform,
                                    init=init)
            self.lin2 = QLinear(hidden_units,dataset.num_classes, num_deg, bit,para_dict=para_list[-1][0], all_positive=True, 
                                    quant_fea=True,
                                    uniform=uniform,
                                    init=init)
            
        else:
            self.lin1 = nn.Linear( hidden_units , hidden_units , bias=True ) 
            self.lin2 = nn.Linear( hidden_units , dataset.num_classes , bias=True )
        
        self.L = 1

    def forward(self, data):
        x, pos, edge_index, edge_weight, batch = data.data, None, data.edge_index, None, None
        #x, pos, edge_index, edge_weight, batch = data, None, None, None, None
        #x = torch.cat((x,pos),dim=1)
        bit_sum=x.new_zeros(1)
     
        if(self.is_q):
            x,_,bit_sum = self.embeding(x,edge_index,bit_sum)
        else:
            x = self.embeding(x)
        for conv in self.convs:
            x,bit_sum = conv(x, edge_index,bit_sum)
        #x = global_mean_pool(x, batch)
        if(self.is_q):
            x,_,bit_sum = self.lin1(x,edge_index,bit_sum)
        else:
            x = self.lin1(x)
        x = F.relu(x)
        if(self.is_q):
            x,_,bit_sum = self.lin2(x,edge_index,bit_sum)
        else:
            x = self.lin2(x)
        return x,bit_sum

In [11]:
import torch.nn.functional as F
import torch.nn as nn


class NormalizedDegree(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, data):
        deg = degree(data.edge_index[0], dtype=torch.float)
        deg = (deg - self.mean) / self.std
        data.x = deg.view(-1, 1)
        return data


def num_graphs(data):
    if data.batch is not None:
        return data.num_graphs
    else:
        return data.data.size(0)
        #return data.x.size(0)



def train(model, optimizer, data,a_loss, a_storage=1):
    model.train()

    total_loss = 0
    #for data in loader:
    optimizer.zero_grad()
    data = data.to(device)
    out,bit_sum = model(data)
    loss = F.cross_entropy(out, data.y.view(-1))
    pred = model(data)[0].max(1)[1]
    correct =pred.eq(data.y.view(-1)).sum().item()
    
    #loss_store = a_loss*F.relu(bit_sum-a_storage)**2
    #loss_store.backward(retain_graph=True)
    loss.backward()
    #otal_loss += loss.item() * data.data.size(0)
    #total_loss += loss.item() * num_graphs(data)
    optimizer.step()
    return loss





def eval_loss(model, data):
    model.eval()

    loss = 0
    #for data in loader:
    data = data.to(device)
    with torch.no_grad():
        out = model(data)[0]
       
    loss = F.cross_entropy(out, data.y.view(-1), reduction="sum").item()

    
    return loss / data.num_nodes





def eval_acc(model, data):
    model.eval()
    #correct = 0
    #for data in loader:
    data = data.to(device)
    with torch.no_grad():
        pred = model(data)[0].max(1)[1]
 
    correct = pred.eq(data.y.view(-1)).sum().item()
    return correct / data.num_nodes


## Load Dataset

In [12]:
from torch_geometric.datasets import Planetoid
import torch

# Load the Cora dataset
dataset = Planetoid(root='./', name='Cora')

In [13]:
folds=5

def k_fold(dataset, folds):
    skf = StratifiedKFold(folds, shuffle=True, random_state=12345)

    test_indices, train_indices = [], []
    X = dataset.data.x  # or whatever your data is
    y = dataset.data.y
    #print('dataset.data.x :', dataset.data.x.shape)
    #print('dataset.data.y :', dataset.data.y.shape)
    for _, idx in skf.split(torch.zeros(len(X)), y):
        test_indices.append(torch.from_numpy(idx))

    val_indices = [test_indices[i % len(test_indices)] for i in range(folds)]


    for i in range(folds):
        train_mask = torch.ones(len(X), dtype=torch.bool)
        train_mask[test_indices[i]] = 0
        train_mask[val_indices[i]] = 0
        train_indices.append(train_mask.nonzero().view(-1))
        
    return train_indices, test_indices, val_indices  



# Convert edge_index to adjacency matrix
adj_matrix = torch.zeros((dataset.data.num_nodes, dataset.data.num_nodes))
adj_matrix[dataset.data.edge_index[0], dataset.data.edge_index[1]] = 1

def build_edge_index(data_idx):
        data_adj_matrix = adj_matrix[data_idx.tolist()][:, data_idx.tolist()]
        # Optionally, convert adjacency matrices back to edge_index tensors
        data_edge_index = torch.nonzero(data_adj_matrix, as_tuple=False)
        row1 = data_edge_index[:, 1]  # Second column
        row2 = data_edge_index[:, 0]  # First column
        # Combine rows to create the desired output tensor
        data_edge_index = torch.stack([row1, row2], dim=0)
        return data_edge_index

C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\data\in_memory_dataset.py:183: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [ ]:
args.max_epoch=100
max_acc = 0.79

accu = []
accu_max = []

for fold, (train_idx, test_idx, val_idx) in enumerate(zip(*k_fold(dataset,5))):
            print_max_acc=0
            train_dataset = dataset.x[train_idx.tolist()]
            train_dataset.y=dataset.y[train_idx.tolist()]
            train_dataset.edge_index =build_edge_index(train_idx)
            train_dataset.num_features=dataset.num_features
            train_dataset.num_classes=dataset.num_classes
            train_dataset.num_nodes=len(train_dataset)

            test_dataset = dataset.x[test_idx.tolist()]
            test_dataset.y=dataset.y[test_idx.tolist()]
            test_dataset.edge_index = build_edge_index(test_idx) 
            test_dataset.num_nodes=len(test_dataset)

            val_dataset = dataset.x[val_idx.tolist()]
            val_dataset.y=dataset.y[val_idx.tolist()]
            val_dataset.edge_index =build_edge_index(val_idx)
            val_dataset.num_nodes=len(val_dataset)


            k=0
            model=GCN(dataset=train_dataset,hidden_units=args.hidden_units,num_layers=args.num_layers,is_q=True,
                num_deg=args.num_deg,
                uniform=args.uniform,
                init = args.init
                ).to(device)
            weight_paras,quant_paras_bit_weight, quant_paras_bit_fea, quant_paras_scale_weight, quant_paras_scale_fea, quant_paras_scale_xw, quant_paras_bit_xw, other_paras = paras_group(model)
            optimizer = torch.optim.Adam([{'params':weight_paras}, 
                                {'params':quant_paras_scale_weight,'lr':args.lr_quant_scale_weight,'weight_decay':0},
                                {'params':quant_paras_scale_fea,'lr':args.lr_quant_scale_fea,'weight_decay':0},
                                {'params':quant_paras_scale_xw,'lr':args.lr_quant_scale_xw,'weight_decay':0},
                                # {'params':quant_paras_bit_weight,'lr':args.lr_quant_bit_weight,'weight_decay':0},
                                {'params':quant_paras_bit_fea,'lr':args.lr_quant_bit_fea,'weight_decay':0},
                                {'params':other_paras}],
                                lr=args.lr, weight_decay=args.weight_decay)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                factor=args.lr_decay_factor,
                                                patience=args.lr_schedule_patience,
                                                verbose=True,threshold=0.0001)
            # if (os.path.exists(path2check)):
            #     model = load_checkpoint(model,path2check)
            for epoch in range(args.max_epoch):
                    t = tqdm(epoch)
                    train_loss=0
                    train_loss = train(model,optimizer,train_dataset,args.a_loss, args.a_storage)

                    start = time.process_time()
                    val_loss = eval_loss(model,val_dataset)
                    end = time.process_time()
                    acc = eval_acc(model,test_dataset)
                 
                    
                    #The below command helps adapt the learning rate during training to improve model performance. 
                    #Without calling it, the learning rate stays at its initial value throughout trainin
                    scheduler.step(val_loss)
                    
                    #The purpose of the following code is to provide real-time updates on training progress,
                   
                    if epoch % 10 == 0:
                        
                        print(f"Eval Epoch: {epoch}| Fold:{fold} |Train_Loss:{train_loss:05.3f}|Val_Loss: {val_loss:05.3f} | Acc: {acc:.3f}")
                              
                            
                            
                    accu.append(acc)
                    if(acc>max_acc):
                        path = path2check+'/'+args.model+'_'+str(hidden_units)+'_on_'+dataset_name+'_'+str(bit)+'bit-'+str(max_epoch)+'.pth.tar'
                        max_acc = acc
                        torch.save({'state_dict': model.state_dict(), 'best_accu': acc,}, path)
                    if(acc>print_max_acc):
                        print_max_acc = acc
                    if (epoch) % 20 == 0:   
                            if(args.resume==True):
                                f = open(file_name,'a')
                                f.write(str(acc))
                                f.write('\n')

            if(args.resume==True):
                    f = open(file_name,'a')
                    f.write('The max accu of the {} runs is:'.format(i))
                    f.write(str(print_max_acc))
                    f.write('\n')           



## Manual Measurement

In [14]:
Eva_final=dict()



AWQ_model_accuracy=[]
T_AWQ_model=[]
Num_parm_AWQ_model=[]
AWQ_model_size=[]



In [15]:
args.max_epoch=100
max_acc = 0.8
for i in range(5):
    print('********************************************')
    print(f'The iteration is :{i+1} ')
    accu = []
    accu_max = []
    Eva=dict()

    for fold, (train_idx, test_idx, val_idx) in enumerate(zip(*k_fold(dataset,5))):
                print_max_acc=0
                train_dataset = dataset.x[train_idx.tolist()]
                train_dataset.y=dataset.y[train_idx.tolist()]
                train_dataset.edge_index =build_edge_index(train_idx)
                train_dataset.num_features=dataset.num_features
                train_dataset.num_classes=dataset.num_classes
                train_dataset.num_nodes=len(train_dataset)

                test_dataset = dataset.x[test_idx.tolist()]
                test_dataset.y=dataset.y[test_idx.tolist()]
                test_dataset.edge_index = build_edge_index(test_idx) 
                test_dataset.num_nodes=len(test_dataset)

                val_dataset = dataset.x[val_idx.tolist()]
                val_dataset.y=dataset.y[val_idx.tolist()]
                val_dataset.edge_index =build_edge_index(val_idx)
                val_dataset.num_nodes=len(val_dataset)

                
                k=0
                model=GCN(dataset=train_dataset,hidden_units=args.hidden_units,num_layers=args.num_layers,is_q=True,
                    num_deg=args.num_deg,
                    uniform=args.uniform,
                    init = args.init
                    ).to(device)
                weight_paras,quant_paras_bit_weight, quant_paras_bit_fea, quant_paras_scale_weight, quant_paras_scale_fea, quant_paras_scale_xw, quant_paras_bit_xw, other_paras = paras_group(model)
                optimizer = torch.optim.Adam([{'params':weight_paras}, 
                                    {'params':quant_paras_scale_weight,'lr':args.lr_quant_scale_weight,'weight_decay':0},
                                    {'params':quant_paras_scale_fea,'lr':args.lr_quant_scale_fea,'weight_decay':0},
                                    {'params':quant_paras_scale_xw,'lr':args.lr_quant_scale_xw,'weight_decay':0},
                                    # {'params':quant_paras_bit_weight,'lr':args.lr_quant_bit_weight,'weight_decay':0},
                                    {'params':quant_paras_bit_fea,'lr':args.lr_quant_bit_fea,'weight_decay':0},
                                    {'params':other_paras}],
                                    lr=args.lr, weight_decay=args.weight_decay)
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                    factor=args.lr_decay_factor,
                                                    patience=args.lr_schedule_patience,
                                                    verbose=True,threshold=0.0001)
                # if (os.path.exists(path2check)):
                #     model = load_checkpoint(model,path2check)
                for epoch in range(args.max_epoch):
                        t = tqdm(epoch)
                        train_loss=0
                        train_loss = train(model,optimizer,train_dataset,args.a_loss, args.a_storage)

                        start = time.process_time()
                        val_loss = eval_loss(model,val_dataset)
                        end = time.process_time()
                        acc = eval_acc(model,test_dataset)
                        # for name,param in model.named_parameters():
                        #     a=param.grad
                        #     if(a!=None):
                        #         writer.add_histogram(tag=name+'_grad', values=a, global_step=epoch)
                        scheduler.step(val_loss)
                        if (epoch) % 20 == 0: 
                            t.set_postfix(
                                        {
                                            "Train_Loss": "{:05.3f}".format(train_loss),
                                            "Val_Loss": "{:05.3f}".format(val_loss),
                                            "Acc": "{:05.3f}".format(acc),
                                            "Epoch":"{:05.1f}".format(epoch),
                                            "Fold":"{:05.1f}".format(fold),
                                        }
                                    )
                        accu.append(acc)
                        if(acc>max_acc):
                            path = path2check+'/'+args.model+'_'+str(hidden_units)+'_on_'+dataset_name+'_'+str(bit)+'bit-'+str(max_epoch)+'.pth.tar'
                            max_acc = acc
                            torch.save({'state_dict': model.state_dict(), 'best_accu': acc,}, path)
                        if(acc>print_max_acc):
                            print_max_acc = acc
                        if(args.resume==True):
                            f = open(file_name,'a')
                            f.write(str(acc))
                            f.write('\n')
                            
                if(args.resume==True):
                        f = open(file_name,'a')
                        f.write('The max accu of the {} runs is:'.format(i))
                        f.write(str(print_max_acc))
                        f.write('\n')
                        

    t0=time.time()
    awq_model_accuracy=eval_acc(model,test_dataset)
    t1=time.time()
    t_awq_model=t1 - t0
    ###
    awq_model_size = get_model_size(model,count_nonzero_only=True)
    num_parm_awq_model=get_num_parameters(model, count_nonzero_only=True)
    ###
    base_model_size=755424
    print(f"awq model has accuracy on test set={awq_model_accuracy:.2f}%")
    print(f"awq model has size={awq_model_size/MiB:.2f} MiB")
    print(f"The time inference of awq model is ={t_awq_model}") 
    print(f"The number of parametrs of awq model is:{num_parm_awq_model}")
    print(f"awq model has size={awq_model_size/MiB:.2f} MiB, "
          f"which is {base_model_size/awq_model_size:.2f} X smaller than "
          f"the {base_model_size/MiB:.2f} MiB base model")
    
    #Update my Eva dictionary
    Eva.update({'awq model accuracy': awq_model_accuracy,
                'time inference of awq model': t_awq_model,
                'number parmameters of awq model': num_parm_awq_model,
                'size of awq model': awq_model_size})  
    
    AWQ_model_accuracy.append(Eva['awq model accuracy'])
    T_AWQ_model.append(Eva['time inference of awq model'])
    Num_parm_AWQ_model.append(int(Eva['number parmameters of awq model']))
    AWQ_model_size.append(int(Eva['size of awq model']))



********************************************
The iteration is :1 


0it [00:01, ?it/s, Train_Loss=2.063, Val_Loss=1.943, Acc=0.140, Epoch=000.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=2.063, Val_Loss=1.943, Acc=0.140, Epoch=000.0, Fold=000.0]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]
0it [00:01, ?it/s, Train_Loss=0.941, Val_Loss=1.525, Acc=0.459, Epoch=020.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.941, Val_Loss=1.525, Acc=0.459, Epoch=020.0, Fold=000.0]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0

Epoch 00059: reducing learning rate of group 0 to 5.0000e-04.
Epoch 00059: reducing learning rate of group 1 to 5.0000e-03.
Epoch 00059: reducing learning rate of group 2 to 5.0000e-03.
Epoch 00059: reducing learning rate of group 3 to 1.0000e-02.
Epoch 00059: reducing learning rate of group 4 to 1.5000e-04.
Epoch 00059: reducing learning rate of group 5 to 5.0000e-04.



0it [00:01, ?it/s]
0it [00:01, ?it/s]
0it [00:01, ?it/s, Train_Loss=0.213, Val_Loss=1.370, Acc=0.526, Epoch=060.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.213, Val_Loss=1.370, Acc=0.526, Epoch=060.0, Fold=000.0]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]


Epoch 00070: reducing learning rate of group 0 to 2.5000e-04.
Epoch 00070: reducing learning rate of group 1 to 2.5000e-03.
Epoch 00070: reducing learning rate of group 2 to 2.5000e-03.
Epoch 00070: reducing learning rate of group 3 to 5.0000e-03.
Epoch 00070: reducing learning rate of group 4 to 7.5000e-05.
Epoch 00070: reducing learning rate of group 5 to 2.5000e-04.


...

Epoch 00078: reducing learning rate of group 0 to 1.2500e-04.
Epoch 00078: reducing learning rate of group 1 to 1.2500e-03.
Epoch 00078: reducing learning rate of group 2 to 1.2500e-03.
Epoch 00078: reducing learning rate of group 3 to 2.5000e-03.
Epoch 00078: reducing learning rate of group 4 to 3.7500e-05.
Epoch 00078: reducing learning rate of group 5 to 1.2500e-04.


0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]
0it [00:01, ?it/s, Train_Loss=0.175, Val_Loss=1.334, Acc=0.556, Epoch=080.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.175, Val_Loss=1.334, Acc=0.556, Epoch=080.0, Fold=004.0]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]


Epoch 00089: reducing learning rate of group 0 to 6.2500e-05.
Epoch 00089: reducing learning rate of group 1 to 6.2500e-04.
Epoch 00089: reducing learning rate of group 2 to 6.2500e-04.
Epoch 00089: reducing learning rate of group 3 to 1.2500e-03.
Epoch 00089: reducing learning rate of group 4 to 1.8750e-05.
Epoch 00089: reducing learning rate of group 5 to 6.2500e-05.



0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]
0it [00:01, ?it/s]

0it [00:01, ?it/s]


Epoch 00100: reducing learning rate of group 0 to 3.1250e-05.
Epoch 00100: reducing learning rate of group 1 to 3.1250e-04.
Epoch 00100: reducing learning rate of group 2 to 3.1250e-04.
Epoch 00100: reducing learning rate of group 3 to 6.2500e-04.
Epoch 00100: reducing learning rate of group 4 to 9.3750e-06.
Epoch 00100: reducing learning rate of group 5 to 3.1250e-05.
awq model has accuracy on test set=0.56%
awq model has size=1.10 MiB
The time inference of awq model is =0.10937094688415527
The number of parametrs of awq model is:287421
awq model has size=1.10 MiB, which is 0.08 X smaller than the 0.09 MiB base model


In [16]:


AWQ_model_accuracy_mean =stat.mean(AWQ_model_accuracy)
AWQ_model_accuracy_std = stat.stdev(AWQ_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'AWQ model accuracy':float(format(AWQ_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of AWQ model accuracy':float(format(AWQ_model_accuracy_std, '.3f'))})
                 

t_AWQ_model_mean = stat.mean(T_AWQ_model)
t_AWQ_model_std =stat.stdev(T_AWQ_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of AWQ model':float(format(t_AWQ_model_mean, '.3f'))})
Eva_final.update({'Std of time inference of AWQ model':float(format(t_AWQ_model_std, '.3f'))})

num_parm_AWQ_model_mean = stat.mean(Num_parm_AWQ_model)
num_parm_AWQ_model_std = stat.stdev(Num_parm_AWQ_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of AWQ model':num_parm_AWQ_model_mean})
Eva_final.update({'Std of number parmameters of AWQ model':num_parm_AWQ_model_std})

AWQ_model_size_mean =stat.mean( AWQ_model_size)
AWQ_model_size_std = stat.stdev(AWQ_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'AWQ model size':AWQ_model_size_mean})
Eva_final.update({'Std of AWQ_model_size':AWQ_model_size_std })


#################################



print(f"All measurement about Quatization process when the number of qGconv layer is equal to two ")   
Eva_final

All measurement about Quatization process when the number of qGconv layer is equal to two 


{'AWQ model accuracy': 0.553,
 'Std of AWQ model accuracy': 0.007,
 'time inference of AWQ model': 0.116,
 'Std of time inference of AWQ model': 0.009,
 'number parmameters of AWQ model': 287421,
 'Std of number parmameters of AWQ model': 0.0,
 'AWQ model size': 9197472,
 'Std of AWQ_model_size': 0.0}

In [17]:
# Create a table when the number of GConv layer is equal to 2.
AWQ_model_accuracy.append(Eva_final['AWQ model accuracy'] )
AWQ_model_accuracy.append(Eva_final['Std of AWQ model accuracy'] )
T_AWQ_model.append(Eva_final['time inference of AWQ model'])
T_AWQ_model.append(Eva_final['Std of time inference of AWQ model'])
Num_parm_AWQ_model.append(Eva_final['number parmameters of AWQ model'] )
Num_parm_AWQ_model.append(Eva_final['Std of number parmameters of AWQ model'])
AWQ_model_size.append(Eva_final['AWQ model size'] )
AWQ_model_size.append(Eva_final['Std of AWQ_model_size'])

In [18]:

table_data = [AWQ_model_accuracy,T_AWQ_model, Num_parm_AWQ_model, AWQ_model_size]

headers=['1', '2', '3', '4','5', 'Mean', 'STD']

tabulate(table_data, headers, tablefmt='fancy_grid')


# New column data
first_column_data =  ['AWQ_model_accuracy', 'T_AWQ_model', 'Num_parm_AWQ_model', 'AWQ_model_size']
# Add a custom index column
table_with_index = tabulate(table_data, headers=['parameters(2 layyers)'] + headers,
                            showindex=first_column_data, tablefmt="fancy_grid", numalign="center")            

# Print the extended table
print(table_with_index)

╒═════════════════════════╤═════════════╤═════════════╤═════════════╤═════════════╤═════════════╤═════════════╤═══════╕
│ parameters(2 layyers)   │      1      │      2      │      3      │      4      │      5      │    Mean     │  STD  │
╞═════════════════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═══════╡
│ AWQ_model_accuracy      │  0.556377   │  0.547135   │  0.543438   │  0.558226   │  0.558226   │    0.553    │ 0.007 │
├─────────────────────────┼─────────────┼─────────────┼─────────────┼─────────────┼─────────────┼─────────────┼───────┤
│ T_AWQ_model             │  0.109371   │  0.124997   │  0.109372   │  0.124996   │  0.109371   │    0.116    │ 0.009 │
├─────────────────────────┼─────────────┼─────────────┼─────────────┼─────────────┼─────────────┼─────────────┼───────┤
│ Num_parm_AWQ_model      │   287421    │   287421    │   287421    │   287421    │   287421    │   287421    │   0   │
├─────────────────────────┼─────────────